# Lab 11 — Hierarchical Sub-Agents and Context Isolation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tertiarycourses/TGS-2020503207-AI-Vibe-Coding-for-Multi-Agents-System/blob/main/labs/lab-11-hierarchical-sub-agents-and-context-isolation/lab-11-hierarchical-sub-agents-and-context-isolation.ipynb)

**Topic:** 5 — MCP and Sub-Agents

**Objective:** Design a hierarchical sub-agent architecture with delegated, isolated context

Build the top of the architecture: a parent agent that delegates bounded tasks to specialised child agents, each with its own context and tools. The children return conclusions, not their whole working trace, which is what keeps a large system reliable.

Full step-by-step instructions are in the Learner Guide.


> **The critical distinction from Lab 06:** a **handoff** transfers control and the specialist produces the final answer, whereas **agents-as-tools** (`agent.as_tool()`) keeps the parent in control, receiving each child's answer back so it can synthesise a combined result. Hierarchical delegation needs the latter, so this lab uses `as_tool()` throughout and never `handoffs=[...]`.

This lab uses the OpenAI Agents SDK. The same architecture is achievable in ADK using `AgentTool`. It also needs the working `server.py` MCP server from Lab 10.


In [ ]:
!pip install -q openai-agents pydantic python-dotenv "mcp[cli]>=1.9,<2" requests


In [ ]:
# API keys: prefer Colab Secrets (key icon in the left sidebar).
# Add each secret there, enable notebook access, then run this cell.
# NEVER paste a key into the notebook - a saved notebook keeps it forever.
import os
from getpass import getpass

try:
    from google.colab import userdata  # available in Colab only
except ImportError:
    userdata = None


def set_key(name: str) -> None:
    """Read a secret from Colab Secrets, falling back to a hidden prompt."""
    if os.environ.get(name):
        return
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass(f"Enter {name}: ")
    os.environ[name] = value


set_key("OPENAI_API_KEY")
print("Keys set:", [v for v in ["OPENAI_API_KEY"] if os.environ.get(v)])


## 1. Bring the Lab 10 MCP server into this session

`MCPServerStdio` spawns the server as a subprocess, so the file must exist on disk with an absolute path. In Colab, re-create it here; locally, point `MCP_SERVER` at your Lab 10 folder.


In [ ]:
%%writefile team_notes.json
{
  "onboarding": "New joiners get a laptop on day 1 and complete security training in week 1.",
  "deployment": "Deploys run Tuesday and Thursday at 14:00 SGT. Fridays are frozen.",
  "oncall": "The on-call rotation changes Monday 09:00. Escalate to the lead after 30 minutes."
}


In [ ]:
%%writefile server.py
"""The Lab 10 MCP server, re-created here so the hierarchy has tools to call."""

import json
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("toolbox")

DATA_FILE = Path(__file__).parent / "team_notes.json"


@mcp.tool()
def current_time(timezone: str = "Asia/Singapore") -> str:
    """Get the current date and time in a given IANA timezone.

    Args:
        timezone: An IANA timezone name, e.g. "Asia/Singapore".
    """
    try:
        now = datetime.now(ZoneInfo(timezone))
    except Exception:
        return f"Error: '{timezone}' is not a valid IANA timezone name."
    return now.strftime("%Y-%m-%d %H:%M:%S %Z")


@mcp.tool()
def search_team_notes(query: str) -> str:
    """Search the internal team handbook for a policy or process.

    Args:
        query: Keywords to search for, e.g. "deployment schedule".
    """
    if not DATA_FILE.exists():
        return "Error: the team notes file is missing."
    try:
        notes = json.loads(DATA_FILE.read_text())
    except json.JSONDecodeError as exc:
        return f"Error: the team notes file is malformed ({exc})."
    terms = query.lower().split()
    hits = [
        f"{topic}: {text}"
        for topic, text in notes.items()
        if any(term in topic.lower() or term in text.lower() for term in terms)
    ]
    return "\n".join(hits) if hits else f"No notes matched '{query}'."


if __name__ == "__main__":
    mcp.run(transport="stdio")


## 2. The bounded contract a sub-agent returns

Instructions alone are unreliable. Context isolation is enforced with an `output_type`, not with polite instructions: a child that ran fifteen tool calls now returns roughly forty tokens to the parent instead of several thousand. **That is the mechanism behind context isolation.**


In [ ]:
from pydantic import BaseModel, Field


class Conclusion(BaseModel):
    """The bounded result a sub-agent returns to its parent."""

    finding: str = Field(description="The conclusion in at most two sentences.")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence from 0 to 1.")
    sources: list[str] = Field(
        default_factory=list, description="Tools or notes used, by name only."
    )


## 3. Specialists without MCP access

Only the research specialist will get the MCP tools. Handing every tool to every agent recreates the overloaded single agent this architecture exists to avoid.


In [ ]:
from agents import Agent, Runner, function_tool

MODEL = "gpt-4o-mini"

analysis_specialist = Agent(
    name="Analysis Specialist",
    instructions=(
        "You analyse figures and identify the single most important pattern. "
        "Return only the conclusion, not your intermediate arithmetic."
    ),
    model=MODEL,
    output_type=Conclusion,
)

writing_specialist = Agent(
    name="Writing Specialist",
    instructions=(
        "You draft concise business prose. Return only the final draft, "
        "with no commentary about your process."
    ),
    model=MODEL,
)

# A specialist that always fails, for the graceful-degradation test.
broken_specialist = Agent(
    name="Broken Specialist",
    instructions="Always fail.",
    model="gpt-4o-mini-nonexistent-model",   # forces an API error
)

research_specialist = None  # bound once the MCP server is connected


## 4. Resilient wrappers, so one failure degrades the result

One broken child must not kill the run. Wrapping a child in a `function_tool` that catches its exceptions converts a fatal failure into a message the orchestrator can act on.


In [ ]:
@function_tool
async def resilient_research(question: str) -> str:
    """Research a focused question. Returns a conclusion, or a failure note.

    Args:
        question: A single clear research question.
    """
    try:
        result = await Runner.run(research_specialist, question, max_turns=5)
        conclusion = result.final_output
        return f"{conclusion.finding} (confidence {conclusion.confidence:.2f})"
    except Exception as exc:
        return (
            f"RESEARCH UNAVAILABLE: {type(exc).__name__}. "
            "Continue without this input and note the gap in your answer."
        )


@function_tool
async def failing_tool(question: str) -> str:
    """A specialist that always fails, for testing degradation.

    Args:
        question: The question to attempt.
    """
    try:
        result = await Runner.run(broken_specialist, question, max_turns=2)
        return str(result.final_output)
    except Exception as exc:
        return f"SPECIALIST UNAVAILABLE: {type(exc).__name__}. Continue without it."


## 5. Build the hierarchy

`agent.as_tool(...)` wraps a whole agent so the parent can invoke it like a function: the child runs its own loop with its own context, and only its final output returns to the parent. The orchestrator's own instructions say it never does the specialist work itself.


In [ ]:
import os
import sys

from agents.mcp import MCPServerStdio

MCP_PYTHON = os.getenv("MCP_PYTHON", sys.executable)
MCP_SERVER = os.getenv("MCP_SERVER", os.path.abspath("server.py"))

ORCHESTRATOR_INSTRUCTIONS = (
    "You are an orchestrator. You never do the specialist work yourself.\n"
    "Break the request into independent sub-tasks, call the appropriate "
    "specialist tool for each, then synthesise their conclusions into one "
    "coherent answer.\n"
    "Call every specialist relevant to the request. If a specialist reports "
    "a failure, continue with the others and note the gap in your answer."
)

mcp_server = MCPServerStdio(
    name="toolbox",
    params={"command": MCP_PYTHON, "args": [MCP_SERVER]},
)
await mcp_server.connect()

research_specialist = Agent(
    name="Research Specialist",
    instructions=(
        "You research a single focused question using your tools. "
        "Return only your conclusion in the required structure. "
        "Never return your working notes or the raw tool output."
    ),
    model=MODEL,
    mcp_servers=[mcp_server],
    output_type=Conclusion,
)

orchestrator = Agent(
    name="Orchestrator",
    instructions=ORCHESTRATOR_INSTRUCTIONS,
    model=MODEL,
    tools=[
        resilient_research,
        analysis_specialist.as_tool(
            tool_name="analyse",
            tool_description="Analyse a set of figures and identify the key pattern.",
        ),
        writing_specialist.as_tool(
            tool_name="draft",
            tool_description="Draft concise business prose from supplied points.",
        ),
        failing_tool,
    ],
)
print("Hierarchy ready with", len(orchestrator.tools), "specialist tools")


## 6. Give the orchestrator a task requiring three sub-agents

The trace should show three specialist tool calls, and the orchestrator itself should perform no specialist work.


In [ ]:
TASK = (
    "Prepare a short briefing for my manager. It needs three things: "
    "(1) our team's deployment schedule, "
    "(2) an analysis of monthly sales 120, 135, 128, 190, 210, and "
    "(3) a 60-word summary of both suitable for an email."
)

result = await Runner.run(orchestrator, TASK)

print("=== Final briefing ===")
print(result.final_output)

print("\n=== Delegation trace ===")
for item in result.new_items:
    name = type(item).__name__
    if "ToolCall" in name:
        print(f"  {name}")


## 7. Run independent sub-agents concurrently

Independent sub-tasks should not run sequentially. Parallel time approximates the slowest single task rather than the sum. Only parallelise genuinely independent work — if the analysis depends on the research result, it must wait.


In [ ]:
import asyncio
import time


async def run_parallel(question_a: str, question_b: str) -> tuple:
    """Run two independent research tasks concurrently."""
    return await asyncio.gather(
        Runner.run(research_specialist, question_a),
        Runner.run(analysis_specialist, question_b),
    )


q_a = "What is the deployment schedule for our team?"
q_b = "Given monthly sales 120, 135, 128, 190, 210, what is the trend?"

start = time.perf_counter()
await Runner.run(research_specialist, q_a)
await Runner.run(analysis_specialist, q_b)
sequential = time.perf_counter() - start

start = time.perf_counter()
await run_parallel(q_a, q_b)
parallel = time.perf_counter() - start

print(f"Sequential: {sequential:.2f}s")
print(f"Parallel:   {parallel:.2f}s")
print(f"Saved:      {sequential - parallel:.2f}s")


## 8. Measure the context reduction

Make the benefit quantitative rather than assumed. Expect a large reduction — multiply that across a dozen sub-agent calls and it is the difference between a system that fits in context and one that does not.


In [ ]:
question = "What is our deployment schedule and on-call policy?"

verbose = Agent(
    name="Verbose Specialist",
    instructions=(
        "Research the question. Show all your working, quote every tool "
        "result in full, and explain your reasoning at length."
    ),
    model=MODEL,
    mcp_servers=[mcp_server],
)
isolated = Agent(
    name="Isolated Specialist",
    instructions="Research the question. Return only your conclusion.",
    model=MODEL,
    mcp_servers=[mcp_server],
    output_type=Conclusion,
)

verbose_result = await Runner.run(verbose, question)
isolated_result = await Runner.run(isolated, question)

verbose_text = str(verbose_result.final_output)
isolated_text = isolated_result.final_output.finding

print(f"Without isolation: {len(verbose_text) // 4} tokens (approx)")
print(f"With isolation:    {len(isolated_text) // 4} tokens (approx)")
print(f"Reduction:         "
      f"{100 * (1 - len(isolated_text) / max(len(verbose_text), 1)):.0f}%")


## 9. Clean up the MCP server

Always `cleanup()` the MCP server — in the `hierarchy.py` script this sits in a `finally` block — or the subprocess is left running.


In [ ]:
await mcp_server.cleanup()
print('MCP server stopped')


## What you learned

- Agents-as-tools keeps the parent in control and lets it synthesise several children's results, whereas a handoff transfers control entirely — hierarchy needs the former.
- Context isolation is enforced with `output_type`, not with polite instructions.
- Each sub-agent should hold only the tools its own job requires; MCP servers attach per agent, not globally.
- `asyncio.gather` parallelises genuinely independent sub-tasks.
- Wrapping a child agent in a try/except tool converts a fatal failure into a degraded result the orchestrator can report.
- Measuring the context reduction turns an architectural claim into evidence.
